<a href="https://www.kaggle.com/code/potlanagasaiprajith/project-fvdd?scriptVersionId=248744447" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

dir_path = '/kaggle/input'
tot_files = 0
folder_counts = []

print("Loading....")
print(f"Total no.of files in the directory are {sum([len(files) for _, _, files in os.walk(dir_path)])}.")

for dirname, _, filenames in os.walk(dir_path):
    total_size = 0
    folder_name = os.path.basename(dirname)
    tot_files += len(filenames)
    if tot_files > 0:
        folder_counts.append((folder_name, tot_files))
    tot_files = 0
    for filename in filenames:
        filepath = os.path.join(dirname, filename)
        try:
            total_size += os.path.getsize(filepath)
        except OSError:
            continue

# Sort folder counts by folder name
folder_counts.sort()

# Print sorted folder counts
for folder_name, count in folder_counts:
    print(f"{folder_name} contains {count} files.")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from torch import nn
import torch.nn.functional as F
from torch import optim
from sklearn.model_selection import train_test_split
print("Compiled Successfully")

In [ ]:
data_path = '/kaggle/input/fruit-and-vegetable-disease-healthy-vs-rotten/Fruit And Vegetable Diseases Dataset'
transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(), 
    transforms.Normalize((0.485,0.456,0.406), (0.229,0.224,0.225))
])
print("Compiled Successfully")

In [ ]:
class_names = sorted(os.listdir(data_path))

for index, name in enumerate(class_names):
    print(f"Class index: {index}, Class name: {name}")

In [ ]:
dataset = datasets.ImageFolder(data_path, transform = transform)

labels = np.array(dataset.targets)
train_indices, test_indices = train_test_split(np.arange(len(labels)), test_size = 0.2, stratify = labels, random_state = 42)
train_indices, valid_indices = train_test_split(train_indices, test_size = 0.1, stratify = labels[train_indices], random_state = 42)                                               

train_data = Subset(dataset, train_indices)
valid_data = Subset(dataset, valid_indices)
test_data = Subset(dataset, test_indices)
train_loader = DataLoader(train_data, batch_size = 32, shuffle = True)
valid_loader = DataLoader(valid_data, batch_size = 32, shuffle = False)
test_loader = DataLoader(test_data, batch_size = 32, shuffle = False)
print("Compiled Successfully")

In [ ]:
print("Train size:",len(train_data))
print("Validation size:",len(valid_data))
print("Test size:",len(test_data))

In [ ]:
class CNN(nn.Module):
    def __init__ (self):
        super().__init__()
        self.network = nn.Sequential(
            
            nn.Conv2d(in_channels = 3, out_channels = 4, kernel_size = 3, stride = 1, padding = 1), 
            nn.ReLU(),
                                                                                                    #256,256,4
            nn.Conv2d(in_channels = 4, out_channels = 8, kernel_size = 3, stride = 1, padding = 1), 
            nn.ReLU(),
                                                                                                    #256,256,8
            nn.Conv2d(in_channels = 8, out_channels = 16, kernel_size = 3, stride = 1, padding = 1), 
            nn.ReLU(), 
            nn.MaxPool2d(kernel_size = 2, stride = 2),
                                                                                                    #128,128,16
            nn.Conv2d(in_channels = 16, out_channels = 32, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2),
                                                                                                    #64,64,32
            nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2),
                                                                                                    #32,32,64
            nn.Conv2d(in_channels = 64, out_channels = 64, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2),
                                                                                                    #16,16,64
            nn.Conv2d(in_channels = 64, out_channels = 64, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2),
                                                                                                    #8,8,64
            nn.Conv2d(in_channels = 64, out_channels = 128, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
                                                                                                    #8,8,128
            nn.Flatten(),
            nn.Linear(128*8*8, 128),
            nn.ReLU(),
            nn.Linear(128, 28)
        )
        
        
    def forward(self, x):
            return self.network(x)
        
print("Compiled Successfully")

In [ ]:
device = torch.device('cuda') # used after activating gpu

model = CNN()
model = model.to(device)    #converting model to gpu from cpu
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 1e-3 )
print("Compiled Successfully")

In [ ]:
import time
for epoch in range(20):
    start_time = time.time()
    for inp,output in train_loader:
        inp, output = inp.to(device), output.to(device)
        pred_out = model(inp)
        loss = criteria(pred_out, output)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    end_time = time.time()
    epoch_time = end_time - start_time
    print(f"loss value of epoch {epoch+1} = {loss} in {epoch_time:.4f} seconds")
    if loss<=0.02:
        break;

In [ ]:
def calculate_accuracy(loader, model):
    correct = 0
    total = 0
    model.eval()
    with torch.no_grad():
        for inp, out in loader:
            inp,out = inp.to(device), out.to(device)
            scores = model(inp)
            _, predictions = scores.max(1)
            correct += (predictions == out).sum().item()
            total += predictions.size(0)
    model.train()
    return correct/total

In [ ]:
print(f"Training Accuracy is {calculate_accuracy(train_loader, model)*100}")
print(f"Validation Accuracy is {calculate_accuracy(valid_loader, model)*100}")
print(f"Testing Accuracy is {calculate_accuracy(test_loader, model)*100}")

In [ ]:
import random

import matplotlib.pyplot as plt
def show_images(images, titles, rows=4, cols=3):
    fig, axes = plt.subplots(rows, cols, figsize=(12, 8))
    for i, (img, title) in enumerate(zip(images, titles)):
        ax = axes[i // cols, i % cols]
        img = img.cpu().numpy().transpose((1, 2, 0))  # Convert to HWC format for displaying and move to CPU
        ax.imshow(img)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# Number of images to display
num_images = 12

# Randomly select indices from the test dataset
indices = random.sample(range(len(test_data)), num_images)
images, titles = [], []

# Set the model to evaluation mode and move to the appropriate device
model.to(device)
model.eval()

# Disable gradient calculation
with torch.no_grad():
    for idx in indices:
        # Get the image and label from the test dataset
        image, label = test_data[idx]
        
        # Move the image to the appropriate device
        image = image.to(device)
        
        images.append(image.cpu())  # Append the image (moved back to CPU for display)
        
        # Get the model's prediction
        output = model(image.unsqueeze(0))
        _, predicted = torch.max(output, 1)
        
        # Get class names
        actual_class = dataset.classes[label]
        predicted_class = dataset.classes[predicted.item()]
        
        # Create a title with predicted and actual class names
        titles.append(f'Pred: {predicted_class}\nActual: {actual_class}')

# Display the images with titles
show_images(images, titles)


In [ ]:
import matplotlib.pyplot as plt

# Data
labels = [
    "Apple_Healthy", "Apple_Rotten", "Banana_Healthy", "Banana_Rotten", "Bellpepper_Healthy",
    "Bellpepper_Rotten", "Carrot_Healthy", "Carrot_Rotten", "Cucumber_Healthy", "Cucumber_Rotten",
    "Grape_Healthy", "Grape_Rotten", "Guava_Healthy", "Guava_Rotten", "Jujube_Healthy",
    "Jujube_Rotten", "Mango_Healthy", "Mango_Rotten", "Orange_Healthy", "Orange_Rotten",
    "Pomegranate_Healthy", "Pomegranate_Rotten", "Potato_Healthy", "Potato_Rotten",
    "Strawberry_Healthy", "Strawberry_Rotten", "Tomato_Healthy", "Tomato_Rotten"
]
values = [
    2438, 2930, 2000, 2800, 611, 591, 620, 580, 608, 593, 200, 200, 200, 200, 200, 200,
    1818, 2247, 2075, 2186, 200, 200, 615, 585, 1603, 1596, 604, 596
]

# Create bar graph
plt.figure(figsize=(14, 8))
plt.bar(labels, values, color='skyblue')
plt.xlabel('Category')
plt.ylabel('Number of Files')
plt.title('Number of Files in Each Category (Healthy vs Rotten)')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()